In [ ]:
!pip install statsmodels

In [ ]:
# 0. EDA

In [ ]:
########################################

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
import matplotlib.pyplot as plt

In [2]:
gaze_df = pd.read_csv('df_final1.csv')
scores_df = pd.read_csv('precalculated_saliency_coverage.csv') 

In [3]:
# 1. PREPARACIÓN DE LAS MÉTRICAS DE ATENCIÓN

# Calculamos la proporción de atención por cada trial
attn_counts = gaze_df.groupby(['participante', 'ImageName', 'main_class']).size().reset_index(name='fix_count')
total_fix = gaze_df.groupby(['participante', 'ImageName']).size().reset_index(name='total_fix')
attn_metrics = attn_counts.merge(total_fix, on=['participante', 'ImageName'])
attn_metrics['attn_prop'] = attn_metrics['fix_count'] / attn_metrics['total_fix']

In [4]:
# 2. PIVOTAR 
df_pivot = attn_metrics.pivot_table(index=['participante', 'ImageName'], 
                                    columns='main_class', values='attn_prop', fill_value=0).reset_index()

In [5]:
df_pivot

main_class,participante,ImageName,ashcan,awning,bag,bannister,base,bench,bicycle,box,...,tank,tower,trade,traffic,tree,truck,van,wall,water,windowpane
0,1,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.137681,0.0,0.000000,0.000000,0.000000,0.0
1,1,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.015556,0.0,0.000000,0.000000,0.000000,0.0
2,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.006912,0.230415,0.0
3,1,7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.040000,0.0,0.000000,0.411111,0.000000,0.0
4,1,8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.048507,0.0,0.192786,0.217662,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,30,141,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.002217,0.0,0.000000,0.088692,0.000000,0.0
1496,30,142,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.000000,0.0,0.004435,0.212860,0.000000,0.0
1497,30,144,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.008869,0.0,0.000000,0.000000,0.000000,0.0
1498,30,148,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.014252,0.111639,0.0,0.000000,0.000000,0.000000,0.0


In [6]:
df_pivot.isnull().sum()

main_class
participante    0
ImageName       0
ashcan          0
awning          0
bag             0
bannister       0
base            0
bench           0
bicycle         0
box             0
bridge          0
building        0
bus             0
car             0
column          0
conveyer        0
door            0
earth           0
fence           0
field           0
floor           0
fountain        0
grass           0
hill            0
house           0
lake            0
minibike        0
mountain        0
palm            0
path            0
person          0
plant           0
pole            0
poster          0
pot             0
railing         0
road            0
rock            0
sea             0
sidewalk        0
signboard       0
sky             0
stairs          0
step            0
streetlight     0
table           0
tank            0
tower           0
trade           0
traffic         0
tree            0
truck           0
van             0
wall            0
water           0

In [7]:
# 3. INTEGRACIÓN CON SCORES
df_final = scores_df.merge(df_pivot, on=['participante', 'ImageName'], how='inner')

df_model = df_final.dropna().reset_index(drop=True)

In [8]:
# 4. SELECCIÓN DE PREDICTORES Y ESTANDARIZACIÓN (Z-SCORE)
# ---------------------------------------------------------

clases_interes = ['ashcan', 'awning', 'bag',
       'bench', 'bicycle', 'box', 'bridge', 'building', 'bus', 'car', 'column',
       'conveyor', 'door', 'fence', 'field', 'floor', 'flowerpot', 'fountain',
       'grass', 'ground', 'handrail', 'hill', 'house', 'lake', 'motorcycle',
       'mountain', 'palm', 'path', 'pedestal', 'person', 'plant', 'pole',
       'poster', 'railing', 'road', 'rock', 'sea', 'sidewalk', 'signboard',
       'sky', 'stair', 'stairs', 'stoplight', 'streetlight', 'table', 'trade',
       'tree', 'truck', 'van', 'wall', 'water', 'windowpane']

predictores_disponibles = []
for clase in clases_interes:
    if clase in df_final.columns:
        col_std = f'{clase}_std'
        # Estandarizamos para poder comparar impactos (Beta weights)
        df_final[col_std] = stats.zscore(df_final[clase])
        predictores_disponibles.append(col_std)

print(f"Predictores estandarizados para el modelo: {predictores_disponibles}")

Predictores estandarizados para el modelo: ['ashcan_std', 'awning_std', 'bag_std', 'bench_std', 'bicycle_std', 'box_std', 'bridge_std', 'building_std', 'bus_std', 'car_std', 'column_std', 'door_std', 'fence_std', 'field_std', 'floor_std', 'fountain_std', 'grass_std', 'hill_std', 'house_std', 'lake_std', 'mountain_std', 'palm_std', 'path_std', 'person_std', 'plant_std', 'pole_std', 'poster_std', 'railing_std', 'road_std', 'rock_std', 'sea_std', 'sidewalk_std', 'signboard_std', 'sky_std', 'stairs_std', 'streetlight_std', 'table_std', 'trade_std', 'tree_std', 'truck_std', 'van_std', 'wall_std', 'water_std', 'windowpane_std']


In [9]:
predictores_disponibles

['ashcan_std',
 'awning_std',
 'bag_std',
 'bench_std',
 'bicycle_std',
 'box_std',
 'bridge_std',
 'building_std',
 'bus_std',
 'car_std',
 'column_std',
 'door_std',
 'fence_std',
 'field_std',
 'floor_std',
 'fountain_std',
 'grass_std',
 'hill_std',
 'house_std',
 'lake_std',
 'mountain_std',
 'palm_std',
 'path_std',
 'person_std',
 'plant_std',
 'pole_std',
 'poster_std',
 'railing_std',
 'road_std',
 'rock_std',
 'sea_std',
 'sidewalk_std',
 'signboard_std',
 'sky_std',
 'stairs_std',
 'streetlight_std',
 'table_std',
 'trade_std',
 'tree_std',
 'truck_std',
 'van_std',
 'wall_std',
 'water_std',
 'windowpane_std']

In [10]:
# 5. MODELO DE EFECTOS MIXTOS 
# ---------------------------------------------------------
if len(predictores_disponibles) > 0:
    # Eliminamos cualquier fila con NaNs y reseteamos el índice para evitar el IndexError
    cols_to_use = ['score', 'participante'] + predictores_disponibles
    if 'grupo' in df_final.columns:
        cols_to_use.append('grupo')
        
    df_model = df_final[cols_to_use].dropna().reset_index(drop=True)

    # Construimos la fórmula dinámicamente
    formula = "score ~ " + " + ".join(predictores_disponibles)
    
    if 'grupo' in df_model.columns:
        formula += " + C(grupo)"

    print(f"Ejecutando modelo con {len(df_model)} observaciones...")

    # Ejecutamos el Mixed Linear Model
    # Usamos df_model que tiene el índice reseteado
    model = smf.mixedlm(formula, data=df_model, groups=df_model["participante"])
    result = model.fit()

    print("\n=== RESULTADOS DEL MODELO MIXTO ===")
    print(result.summary())
    
    # Guardar resultados para el paper
    summary_df = result.summary().tables[1]
    summary_df.to_csv("mixed_effects_results.csv")
else:
    print("Error: No se encontraron las clases especificadas en los datos.")

Ejecutando modelo con 1500 observaciones...

=== RESULTADOS DEL MODELO MIXTO ===
          Mixed Linear Model Regression Results
Model:              MixedLM Dependent Variable: score     
No. Observations:   1500    Method:             REML      
No. Groups:         30      Scale:              2.8365    
Min. group size:    50      Log-Likelihood:     -3021.4915
Max. group size:    50      Converged:          Yes       
Mean group size:    50.0                                  
----------------------------------------------------------
                Coef.  Std.Err.   z    P>|z| [0.025 0.975]
----------------------------------------------------------
Intercept        5.091    0.155 32.767 0.000  4.787  5.396
ashcan_std       0.133    0.044  2.982 0.003  0.045  0.220
awning_std       0.084    0.051  1.670 0.095 -0.015  0.183
bag_std          0.041    0.048  0.867 0.386 -0.052  0.135
bench_std       -0.135    0.047 -2.844 0.004 -0.228 -0.042
bicycle_std     -0.032    0.046 -0.698 0.485 

In [ ]:
#### GRAFICAR LOS RESULTADOS

In [ ]:
#df = pd.read_csv("mixed_effects_results.csv")

In [ ]:
df.columns

In [11]:
from mixed_table_generator import generate_tables

In [12]:
generate_tables(result, mode="normal")

PDFs generados correctamente ✅🔥


In [13]:
generate_tables(result, mode="filter")


PDFs generados correctamente ✅🔥


In [ ]:
#####################
#   CREAR DATASET   #
#####################

In [ ]:
import pandas as pd
import json

# Cargar el JSON
with open("data_hololens.json", "r") as f:
    data = json.load(f)

# Construir las filas del DataFrame
rows = []
for key in range(150):
    entry = data[str(key)]
    scores = [p["score"] for p in entry["score_participant"]]
    score_mean = sum(scores) / len(scores)
    for p in entry["score_participant"]:
        rows.append({
            "id1": entry["ID"],
            "id2": key,
            "participante": p["participant"],
            "score": p["score"],
            "score_mean": score_mean
        })

df = pd.DataFrame(rows)

In [ ]:
df.head(20)

In [ ]:
df.to_csv("data_hololens.csv", index=False)